<a href="https://colab.research.google.com/github/9lenrg/medley_vox/blob/main/MedleyVox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Inference for MedleyVox

Medley Vox is a [dataset for testing algorithms for separating multiple singers](https://arxiv.org/pdf/2211.07302) within a single music track. Also, the [authors of Medley Vox](https://github.com/jeonchangbin49/MedleyVox) proposed a neural network architecture for separating singers. However, unfortunately, they did not publish the weights. Later, their training process was [repeated by Cyru5](https://huggingface.co/Cyru5/MedleyVox/tree/main), who trained several models and published the weights in the public domain. Now this WebUI is created to use the trained models and weights for inference. Here are some precautions:
1. Put the [downloaded models](https://huggingface.co/Cyru5/MedleyVox) in the 'checkpoints' folder in folder format, with each model folder containing a model file (.pth) and its corresponding configuration file (.json).
2. If you use overlapadd and the choice of model is 'w2v' or 'w2v_chunk', you need to download the pretrained model [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) and put it in the 'pretrained' folder.
3. At present, the audio output sampling rate supported by the model is 24000kHz and cannot be changed. To solve this, you can use [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo), or [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) for audio super-resolution.
4. When using WebUI on cloud platforms or Colab, please place the audio to be processed in the 'inputs' folder, and the processing results will be stored in the 'results' folder. The 'Select folder' and 'Open folder' buttons are invalid in the cloud.
5. If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use 'use_overlapadd'. Among the 'use_overlapadd' options, "ola", "ola_norm", and "w2v" all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with 'vad_method' options spec and webrtc when using either of the "_chunk" methods. Chunking has proven to be very useful therefore it is on by default.

# Initialize environment

In [1]:
#@title Clone repository and install requirements
#@markdown # Clone repository and install requirements
#@markdown

!nvidia-smi
!git clone https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI
%cd /content/MedleyVox-Inference-WebUI
!python -m pip install --upgrade pip==24.0 setuptools
!python -m pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124
!mkdir -p inputs
!mkdir -p results

!mkdir -p "/content/MedleyVox-Inference-WebUI/checkpoint/vocals_238"
%cd "/content/MedleyVox-Inference-WebUI/checkpoint/vocals_238"
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/vocals%20238/vocals.pth
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/vocals%20238/vocals.json
!mkdir -p "/content/MedleyVox-Inference-WebUI/checkpoint/multi_singing_librispeech_138"
%cd "/content/MedleyVox-Inference-WebUI/checkpoint/multi_singing_librispeech_138"
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/multi_singing_librispeech_138/vocals.pth
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/multi_singing_librispeech_138/vocals.json
!mkdir -p "/content/MedleyVox-Inference-WebUI/checkpoint/singing_librispeech_ft_iSRNet"
%cd "/content/MedleyVox-Inference-WebUI/checkpoint/singing_librispeech_ft_iSRNet"
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/singing_librispeech_ft_iSRNet/vocals.pth
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/singing_librispeech_ft_iSRNet/vocals.json
!mkdir -p "/content/MedleyVox-Inference-WebUI/pretrained"
%cd "/content/MedleyVox-Inference-WebUI/pretrained"
!wget https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt
%cd /content/MedleyVox-Inference-WebUI

Sat Apr 18 02:08:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Inference

### Place the audio to be processed in the 'inputs' folder, and the processing results will be stored in the 'results' folder. There are two ways to tun inference: use WebUI or use command line.

- Use WebUI: Run the WebUI startup code block and then access the WebUI through the public link.
- Use command line: Select appropriate inference parameters and run the the command line code block.

### Explanation of reasoning parameters. For more infrmation, refer to `inference.py`.

- `Model name`: Select which model you want to use.
- `Use overlapadd`: Use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.
- `Separate storage`: Save results in separate folders with the same name as the input file.
- `Output format`: Select the output format of the results.
- `VAD method`: What method do you want to use for 'voice activity detection (vad) -- split chunks -- processing. Only valid when 'w2v_chunk' or 'sf_chunk' for args.use_overlapadd.
- `Spectral features`: What spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)
- `OLA window length`: OLA window size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).
- `OLA hop length`: OLA hop size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).
- `Wav2Vec nth layer output`: Wav2Vec nth layer output, only valid when using w2v or w2v_chunk. For example: 0 1 2 3, default: 0
- `Use EMA model`: Use EMA model or online model? Only vaind when args.ema it True (model trained with EMA).
- `Mix consistent output`: Only valid when the model is trained with mixture_consistency loss.
- `Reorder chunks`: OLA reorder chunks. Only valid when using ola or ola_norm.
- `Skip error files`: Skip error files while separating instead of stopping.

If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use `use_overlapadd`. Among the `use_overlapadd` options, "ola", "ola_norm", and "w2v" all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with `vad_method` options spec and webrtc when using either of the "_chunk" methods. Chunking has proven to be very useful therefore it is on by default.

In [3]:
!pip install --upgrade librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 7.7 MB/s eta 0:00:00
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Attempting uninstall: librosa
    Found existing installation: librosa 0.9.1
    Uninstalling librosa-0.9.1:
      Successfully uninstalled librosa-0.9.1


In [5]:
!pip install torch==2.5.1+cu124 torchaudio==2.5.1+cu124 torchvision==0.20.1+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 49.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 119.3 MB/s eta 0:00:00
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.25.0+cu128
    Uninstalling torchvision-0.25.0+cu128:
      Successfully uninstalled torchvision-0.25.0+cu128
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0
    Uninstalling torchaudio-2.11.0:
      Successfully uninstalled torchaudio-2.11.0


In [9]:
!grep -i "checkpoint" /content/MedleyVox-Inference-WebUI/webui.py
!grep -i "dropdown" /content/MedleyVox-Inference-WebUI/webui.py
!grep -i "os.listdir" /content/MedleyVox-Inference-WebUI/webui.py

os.makedirs("checkpoints", exist_ok=True)
MODEL_DIR = "checkpoints"
        gr.Markdown(value=i18n("Medley Vox is a [dataset for testing algorithms for separating multiple singers](https://arxiv.org/pdf/2211.07302) within a single music track. Also, the [authors of Medley Vox](https://github.com/jeonchangbin49/MedleyVox) proposed a neural network architecture for separating singers. However, unfortunately, they did not publish the weights. Later, their training process was [repeated by Cyru5](https://huggingface.co/Cyru5/MedleyVox/tree/main), who trained several models and published the weights in the public domain. Now this WebUI is created to use the trained models and weights for inference. Here are some precautions:<br>1. Put the [downloaded models](https://huggingface.co/Cyru5/MedleyVox) in the 'checkpoints' folder in folder format, with each model folder containing a model file (.pth) and its corresponding configuration file (.json).<br>2. If you use overlapadd and the choice of 

In [10]:
!mv /content/MedleyVox-Inference-WebUI/checkpoint /content/MedleyVox-Inference-WebUI/checkpoints

In [12]:
# Memindahkan semua model keluar ke folder yang benar
!mv /content/MedleyVox-Inference-WebUI/checkpoints/checkpoint/* /content/MedleyVox-Inference-WebUI/checkpoints/

# Menghapus folder 'checkpoint' kosong yang sudah tidak terpakai
!rm -rf /content/MedleyVox-Inference-WebUI/checkpoints/checkpoint

In [14]:
import re

# Lokasi file library fairseq yang error di dalam Colab
file_path = '/usr/local/lib/python3.12/dist-packages/fairseq/dataclass/configs.py'

try:
    with open(file_path, 'r') as f:
        content = f.read()

    # Menambahkan fungsi 'field' yang dibutuhkan Python 3.12
    if 'from dataclasses import dataclass, field' not in content:
        content = content.replace('from dataclasses import dataclass', 'from dataclasses import dataclass, field')
        if 'from dataclasses import dataclass, field' not in content:
            content = 'from dataclasses import field\n' + content

    # Mencari kode lama yang dilarang dan menggantinya dengan format baru
    # Contoh: common: CommonConfig = CommonConfig() -> common: CommonConfig = field(default_factory=CommonConfig)
    patched_content = re.sub(r'(\w+):\s*([A-Za-z0-9_]+)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', content)

    with open(file_path, 'w') as f:
        f.write(patched_content)

    print("✅ Library 'fairseq' berhasil ditambal untuk Python 3.12!")
except Exception as e:
    print(f"Gagal menambal: {e}")

✅ Library 'fairseq' berhasil ditambal untuk Python 3.12!


In [16]:
import re

# Lokasi file library hydra yang error di dalam Colab
file_path = '/usr/local/lib/python3.12/dist-packages/hydra/conf/__init__.py'

try:
    with open(file_path, 'r') as f:
        content = f.read()

    # Menambahkan fungsi 'field' yang dibutuhkan Python 3.12
    if 'from dataclasses import dataclass, field' not in content:
        content = content.replace('from dataclasses import dataclass', 'from dataclasses import dataclass, field')
        if 'from dataclasses import dataclass, field' not in content:
            content = 'from dataclasses import field\n' + content

    # Mencari kode lama yang dilarang dan menggantinya dengan format baru
    patched_content = re.sub(r'(\w+):\s*([A-Za-z0-9_]+)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', content)

    with open(file_path, 'w') as f:
        f.write(patched_content)

    print("✅ Library 'hydra' berhasil ditambal untuk Python 3.12!")
except Exception as e:
    print(f"Gagal menambal: {e}")

✅ Library 'hydra' berhasil ditambal untuk Python 3.12!


In [18]:
!pip install omegaconf==2.3.0 hydra-core==1.3.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.4 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=392ff286ff2bb5a770892f32dbcf4b5f39ae56892dcee7fb8a4cb5760c604e1e
  Stored in directory: /root/.cache/pip/wheels/1f/be/48/13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfully built antlr4-python3-runtime
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.8
    Uninstalling antlr4-python3-runtime-4.8:
      Successfully uninstalled antlr4-python3-runtime-4.8
  Attempting uninstall: omegaconf
    Found existing installation: omegaconf 2.0.6
    Uninstalling omegaconf-2.0.6:
      Successfully uninstalled omegaconf-2.0.6
  Attempting u

In [2]:
import os

def apply_ultimate_patch():
    # 1. Nonaktifkan inisialisasi awal Hydra di Fairseq yang menyebabkan crash
    fairseq_init = '/usr/local/lib/python3.12/dist-packages/fairseq/__init__.py'
    if os.path.exists(fairseq_init):
        with open(fairseq_init, 'r') as f:
            content = f.read()

        if 'hydra_init()' in content and '# hydra_init()' not in content:
            content = content.replace('hydra_init()', '# hydra_init()')
            with open(fairseq_init, 'w') as f:
                f.write(content)

    # 2. Perbaiki bug identitas _MISSING_TYPE Python 3.12 di dalam OmegaConf
    base_dir = '/usr/local/lib/python3.12/dist-packages/omegaconf'
    for filename in ['_utils.py', 'omegaconf.py']:
        filepath = os.path.join(base_dir, filename)
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                content = f.read()

            # Mengganti pengecekan usang dengan pengecekan aman ala Python 3.12
            replacements = {
                'f.default is not dataclasses.MISSING': 'type(f.default).__name__ != "_MISSING_TYPE"',
                'f.default is dataclasses.MISSING': 'type(f.default).__name__ == "_MISSING_TYPE"',
                'f.default_factory is not dataclasses.MISSING': 'type(f.default_factory).__name__ != "_MISSING_TYPE"',
                'f.default_factory is dataclasses.MISSING': 'type(f.default_factory).__name__ == "_MISSING_TYPE"',
                'getattr(f, "default_factory", dataclasses.MISSING) is not dataclasses.MISSING': 'type(getattr(f, "default_factory", dataclasses.MISSING)).__name__ != "_MISSING_TYPE"'
            }

            for old, new in replacements.items():
                content = content.replace(old, new)

            with open(filepath, 'w') as f:
                f.write(content)

apply_ultimate_patch()
print("✅ Tambalan Pamungkas berhasil diterapkan pada sistem!")

✅ Tambalan Pamungkas berhasil diterapkan pada sistem!


In [4]:
import os
import re

fairseq_dir = '/usr/local/lib/python3.12/dist-packages/fairseq'
# Mencari pola yang dilarang Python 3.12: variabel: Kelas = Kelas()
pattern = re.compile(r'(\s+)([A-Za-z0-9_]+):\s*([A-Za-z0-9_]+)\s*=\s*\3\(\)')

patched_count = 0

print("Memulai pemindaian dan penambalan massal...")

for root, _, files in os.walk(fairseq_dir):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            with open(filepath, 'r') as f:
                content = f.read()

            if pattern.search(content):
                # Memastikan fungsi 'field' diimpor dengan benar
                if 'from dataclasses import' in content and 'field' not in content:
                    content = re.sub(r'from dataclasses import (.*)', r'from dataclasses import \1, field', content, count=1)
                elif 'from dataclasses import field' not in content and 'import dataclasses' not in content:
                    content = 'from dataclasses import field\n' + content

                # Mengganti semua kode usang dengan format baru
                new_content = pattern.sub(r'\1\2: \3 = field(default_factory=\3)', content)

                with open(filepath, 'w') as f:
                    f.write(new_content)
                print(f"🔧 Diperbaiki: {file}")
                patched_count += 1

print(f"✅ Selesai! Berhasil menambal total {patched_count} file yang bermasalah di dalam fairseq.")

Memulai pemindaian dan penambalan massal...
🔧 Diperbaiki: transformer_config.py
🔧 Diperbaiki: data2vec_text.py
🔧 Diperbaiki: infer.py
🔧 Diperbaiki: w2vu_generate.py
🔧 Diperbaiki: unpaired_audio_text.py
🔧 Diperbaiki: wav2vec_u.py
✅ Selesai! Berhasil menambal total 6 file yang bermasalah di dalam fairseq.


In [7]:
!pip install git+https://github.com/facebookresearch/fairseq.git

  Cloning https://github.com/facebookresearch/fairseq.git to /tmp/pip-req-build-n928mqen
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/fairseq.git /tmp/pip-req-build-n928mqen
  Resolved https://github.com/facebookresearch/fairseq.git to commit 3d262bb25690e4eb2e7d3c1309b1e9c406ca4b99
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached hydra_core-1.0.7-py3-none-any.whl.metadata (3.7 kB)
  Using cached omegaconf-2.0.6-py3-none-any.whl.metadata (3.0 kB)
  Using cached antlr4_python3_runtime-4.8-py3-none-any.whl
Using cached hydra_core-1.0.7-py3-none-any.whl (123 kB)
Using cached omegaconf-2.0.6-py3-none-any.whl (36 kB)
  Created wheel for fairseq: filename=fairseq-0.12.2-cp312-cp312-linux_x86_64.whl size=21466479 sha256=a6e1491a7f2a929fbec4f481b7bd2974bfd66675c6b650de6795f9986ee767b1
  St

In [9]:
# 1. Kembalikan versi library yang sempat diturunkan paksa oleh fairseq
!pip install omegaconf==2.3.0 hydra-core==1.3.2

# 2. Retas file inti sistem Python 3.12 di Google Colab
import re
path = '/usr/lib/python3.12/dataclasses.py'
try:
    with open(path, 'r') as f:
        code = f.read()

    # Cari kode yang selalu melempar error "mutable default" dan matikan (diubah jadi 'pass')
    code = re.sub(
        r"raise ValueError\(f'mutable default.*?use default_factory'\)",
        r"pass  # Fitur keamanan ini dimatikan paksa demi fairseq",
        code,
        flags=re.DOTALL
    )

    with open(path, 'w') as f:
        f.write(code)
    print("\n✅ Sistem inti Python 3.12 berhasil diretas! Semua rintangan telah dibersihkan.")
except Exception as e:
    print(f"\n❌ Gagal: {e}")

  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached antlr4_python3_runtime-4.9.3-py3-none-any.whl
Using cached omegaconf-2.3.0-py3-none-any.whl (79 kB)
Using cached hydra_core-1.3.2-py3-none-any.whl (154 kB)
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.8
    Uninstalling antlr4-python3-runtime-4.8:
      Successfully uninstalled antlr4-python3-runtime-4.8
  Attempting uninstall: omegaconf
    Found existing installation: omegaconf 2.0.6
    Uninstalling omegaconf-2.0.6:
      Successfully uninstalled omegaconf-2.0.6
  Attempting uninstall: hydra-core
    Found existing installation: hydra-core 1.0.7
    Uninstalling hydra-core-1.0.7:
      Successfully uninstalled hydra-core-1.0.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following d


✅ Sistem inti Python 3.12 berhasil diretas! Semua rintangan telah dibersihkan.


In [2]:
# Memperbaiki library webrtcvad agar tidak memanggil pkg_resources yang usang
file_path = '/usr/local/lib/python3.12/dist-packages/webrtcvad.py'

try:
    with open(file_path, 'r') as f:
        content = f.read()

    # Mematikan import pkg_resources dan mengganti pengecekan versi otomatis dengan teks statis
    content = content.replace('import pkg_resources', '# import pkg_resources')
    content = content.replace("pkg_resources.get_distribution('webrtcvad').version", "'2.0.10'")

    with open(file_path, 'w') as f:
        f.write(content)

    print("✅ Library 'webrtcvad' berhasil ditambal!")
except Exception as e:
    print(f"❌ Gagal menambal: {e}")

✅ Library 'webrtcvad' berhasil ditambal!


In [ ]:
#@title Run inference in WebUI
#@markdown # Run inference in WebUI
#@markdown

#@markdown

#@markdown Language Setting
language = "English" #@param ["English", "简体中文"]

import os
language_dict = {"English": "en_US", "简体中文": "zh_CN"}
os.environ["LANGUAGE"] = language_dict[language]

%cd /content/MedleyVox-Inference-WebUI
!python webui.py -s

/content/MedleyVox-Inference-WebUI
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://bfb59e927ce28ec637.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Inference started... Click 'Forced stop inference' to stop the process.
Inference process started, PID: 29704.
/usr/bin/python3 inference.py --target "vocals" --exp_name "singing_librispeech_ft_iSRNet" --model_dir "checkpoints" --use_gpu y --use_overlapadd ola --vad_method spec --spectral_features mfcc --w2v_ckpt_dir pretrained --w2v_nth_layer_output 0 --use_ema_model y --mix_consistent_out y --reorder_chunks y --skip_error y --separate_storage y --output_format wav --inference_data_dir "temp" --results_save_dir "results/"
2026-04-18 03:41:55.393771: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT 

In [ ]:
#@title Run inference in Command Line
#@markdown # Run inference in Command Line
#@markdown

#@markdown

#@markdown File and model Parameters
folder_input = "inputs" #@param {type:"string"}
store_dir = "results" #@param {type:"string"}
model_name = "vocals 238" #@param ["vocals_238", "multi_singing_librispeech_138", "singing_librispeech_ft_iSRNet"]

#@markdown

#@markdown Common Parameters
use_overlapadd = "ola" #@param ["None", "ola", "ola_norm", "w2v", "w2v_chunk", "sf_chunk"]
separate_storage = False #@param {type:"boolean"}
skip_error = True #@param {type:"boolean"}
output_format = "wav" #@param ["wav", "flac", "mp3"]

#@markdown

#@markdown Advanced Parameters
vad_method = "spec" #@param ["spec", "webrtc"]
spectral_features = "mfcc" #@param ["mfcc", "spectral_centroid"]
ola_window_len = "0" #@param {type:"string"}
ola_hop_len = "0" #@param {type:"string"}
w2v_nth_layer_output = "0" #@param {type:"string"}
use_ema_model = True #@param {type:"boolean"}
mix_consistent_out = True #@param {type:"boolean"}
reorder_chunks = True #@param {type:"boolean"}

import os
import glob

MODEL_DIR = "checkpoint"
PRETRAINED_MODEL_DIR = "pretrained"
use_gpu = True

model_file = os.path.basename(glob.glob(os.path.join(MODEL_DIR, model_name, "*.pth"))[0])
target = model_file.replace(".pth", "")
exp_name = model_name
model_dir = MODEL_DIR
params = f"--target \"{target}\" --exp_name \"{exp_name}\" --model_dir \"{model_dir}\""
if use_gpu:
    params += " --use_gpu y"
else:
    params += " --use_gpu n"
if use_overlapadd != "None":
    params += f" --use_overlapadd {use_overlapadd}"
params += f" --vad_method {vad_method} --spectral_features {spectral_features} --w2v_ckpt_dir {PRETRAINED_MODEL_DIR} --w2v_nth_layer_output {w2v_nth_layer_output}"
if ola_window_len != "0":
    params += f" --ola_window_len {ola_window_len}"
if ola_hop_len != "0":
    params += f" --ola_hop_len {ola_hop_len}"
if use_ema_model:
    params += " --use_ema_model y"
else:
    params += " --use_ema_model n"
if mix_consistent_out:
    params += " --mix_consistent_out y"
else:
    params += " --mix_consistent_out n"
if reorder_chunks:
    params += " --reorder_chunks y"
else:
    params += " --reorder_chunks n"
if skip_error:
    params += " --skip_error y"
else:
    params += " --skip_error n"
if separate_storage:
    params += f" --separate_storage y"
else:
    params += f" --separate_storage n"
params += f" --output_format {output_format} --inference_data_dir \"{folder_input}\" --results_save_dir \"{store_dir}\""
print(params)

%cd /content/MedleyVox-Inference-WebUI
!python inference.py {params}